# Gemma 4 E2B Fine-Tuning for Edge Vision Demo

Fine-tunes Gemma 4 E2B to respond tersely to 4 vision tasks:
- **Detect**: object labels as CSV
- **Read**: OCR text extraction
- **Describe**: one-sentence scene summary
- **Safety**: safe/alert judgment

**Requirements**: GPU with 16GB+ VRAM (Colab T4 works)

**Steps**:
1. Upload `training_data/train.jsonl` and `recordings/` folder
2. Run all cells
3. Download the GGUF file and deploy to Jetson

In [ ]:
# Install dependencies
!pip install -q unsloth trl pillow datasets

In [ ]:
# Upload training data
# Option A: Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# TRAINING_DATA = '/content/drive/MyDrive/gemma4-demo/training_data/train.jsonl'
# RECORDINGS_DIR = '/content/drive/MyDrive/gemma4-demo/recordings'

# Option B: Direct upload
# from google.colab import files
# uploaded = files.upload()  # Upload train.jsonl

# Option C: Already on disk (if running on own GPU)
TRAINING_DATA = 'training_data/train.jsonl'
RECORDINGS_DIR = 'recordings'

In [ ]:
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    model_name="google/gemma-4-E2B-it",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules="all-linear",
)

print(f"Model loaded. Trainable params: {model.print_trainable_parameters()}")

In [ ]:
import json
from PIL import Image
from datasets import Dataset

records = []
skipped = 0
for line in open(TRAINING_DATA):
    rec = json.loads(line)
    img_path = rec["image_path"]
    
    # Handle path remapping if running on a different machine
    if not os.path.exists(img_path):
        # Try relative path
        img_path = os.path.join(RECORDINGS_DIR, '/'.join(img_path.split('/')[-2:]))
    
    if not os.path.exists(img_path):
        skipped += 1
        continue
    
    try:
        img = Image.open(img_path).convert("RGB")
    except Exception:
        skipped += 1
        continue
    
    records.append({
        "image": img,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": rec["prompt"]},
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": rec["response"]},
                ],
            },
        ],
    })

print(f"Loaded {len(records)} training examples, skipped {skipped}")
dataset = Dataset.from_list(records)

In [ ]:
import os
from trl import SFTTrainer, SFTConfig
from unsloth import UnslothVisionDataCollator

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="gemma4-demo-checkpoints",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=10,
        save_steps=100,
        fp16=True,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
    ),
)

stats = trainer.train()
print(f"\nTraining loss: {stats.training_loss:.4f}")
print(f"Training time: {stats.metrics.get('train_runtime', 0):.0f}s")

In [ ]:
# Test the fine-tuned model before export
FastVisionModel.for_inference(model)

test_img = Image.open(records[0]["image_path"] if "image_path" in records[0] else 
                      os.path.join(RECORDINGS_DIR, "box/frame_000100.jpg")).convert("RGB")

prompts = [
    "Which are visible? person, box, bottle, sign. CSV only, no explanation.",
    "Read all text visible in this image. Return only the text strings, comma-separated. Nothing else.",
    "Describe this scene in one sentence, 15 words max.",
    "Is this scene safe? Reply: safe OR alert: [reason]. Nothing else.",
]

for prompt in prompts:
    messages = [{"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": prompt},
    ]}]
    
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        tokenize=True, return_tensors="pt",
        return_dict=True,
    ).to(model.device)
    inputs["images"] = [test_img]
    
    outputs = model.generate(**inputs, max_new_tokens=32, temperature=0.1)
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"{prompt[:40]}...")
    print(f"  → {response}")
    print()

In [ ]:
# Export to GGUF for deployment on Jetson
OUTPUT_DIR = "gemma4-demo-tuned"

model.save_pretrained_gguf(
    OUTPUT_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)

import os
print(f"\nOutput files in {OUTPUT_DIR}/")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024 / 1024
    print(f"  {f} ({size:.1f} MB)")

print("\nDeploy to Jetson:")
print(f"  scp {OUTPUT_DIR}/*.gguf jetson:~/models/gemma4-demo/")
print("  Then update start-server.sh --model path")

In [ ]:
# Download the GGUF (Colab only)
# from google.colab import files
# import glob
# for f in glob.glob(f"{OUTPUT_DIR}/*.gguf"):
#     files.download(f)